# Feature Engineering & Data Quality
### Titanic: Machine Learning from Disaster

**Objective:** Improve the predictive signal for passenger survival through hypothesis-driven feature engineering, and audit the dataset for quality issues that could make a model unreliable or biased.

**Target:** `Survived` (1 = survived, 0 = did not survive), a binary classification task.

**Dataset:** Kaggle `train.csv`, 891 passengers. Kaggle's `test.csv` is not used because it has no labels; instead, a held-out test set is created from `train.csv`.

**Notebook structure**
1. Setup & data loading
2. Data-quality audit
3. Leakage & class imbalance checks
4. Train/test split & baseline pipeline
5. Feature engineering (with hypotheses)
6. Engineered model
7. Before/after comparison & hypothesis evaluation
8. Conclusion

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, f1_score, classification_report

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv("data/train.csv")
print("Shape:", df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


### Column dictionary

| Column | Meaning |
|---|---|
| PassengerId | Row identifier (no predictive meaning) |
| Survived | Target: 1 = survived, 0 = died |
| Pclass | Ticket class (1 = upper, 2 = middle, 3 = lower) |
| Name | Full name, including title (Mr, Mrs, Miss, Master…) |
| Sex | male / female |
| Age | Age in years |
| SibSp | Number of siblings/spouses aboard |
| Parch | Number of parents/children aboard |
| Ticket | Ticket number |
| Fare | Ticket price |
| Cabin | Cabin number |
| Embarked | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) |